In [1]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier

In [2]:
import numpy as np
import pandas as pd


In [24]:
df=pd.read_csv('tested.csv')
df.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q


In [5]:
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'],inplace=True)



In [6]:
x_train, x_test, y_train, y_test = train_test_split(df.drop(columns=['Survived']),
                                                   df['Survived'],
                                                   test_size=0.2,
                                                   random_state=42)


In [7]:
x_train.head(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
336,2,male,32.0,0,0,13.0,S
31,2,male,24.0,2,0,31.5,S


In [8]:
x_train.isnull().sum() #to check the null values in data

,0
Pclass,0
Sex,0
Age,72
SibSp,0
Parch,0
Fare,1
Embarked,0


Imputation -First step in pipeline


In [9]:
trf1 = ColumnTransformer(
   [('age_imputer', SimpleImputer(), [2]),
   ('fare_imputer', SimpleImputer(strategy='most_frequent'),[6])],remainder='passthrough')
# remainder = passthrough is This is VERY important.
# Without it:
# only transformed columns are kept
# all other columns are dropped

OneHot Encoding for Sex and Embarked column

In [10]:
trf2 = ColumnTransformer([
    (
        'ohe_sex_emb',
        OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
        [1,6]
    )
], remainder='passthrough')


Scaling

In [11]:
trf3= ColumnTransformer(
    [('scaling', MinMaxScaler(), slice(0,10))])
#  we are taking slice 0-10 as after ohe encoding 5 more columns should be added( male=2, embarked=3)

In [12]:
# Feature selection
trf4 = SelectKBest(score_func=chi2,k=8)

Creating decision tree object

In [13]:
trf5 = DecisionTreeClassifier()

# **Creating Pipeline**


In [14]:
pipe = Pipeline([
    ('trf1',trf1),
    ('trf2',trf2),
    ('trf3',trf3),
    ('trf4',trf4),
    ('trf5',trf5)
]) # Used the Pipeline instaed of make_pipeline coz when debugging it helps

In [15]:
# train the pipeline

pipe.fit(x_train, y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('age_imputer',
                                                  SimpleImputer(), [2]),
                                                 ('fare_imputer',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_emb',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scaling', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x7cab9b3ead40>)),
                ('trf5', DecisionTreeClassifier())])

In [16]:
# Exploring the pipeline

pipe.named_steps

# As here output is dict which shows the key value pair of every Column Transformer object, reason to use pipeline class instaed of make_pipeline

{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('age_imputer', SimpleImputer(), [2]),
                                 ('fare_imputer',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'trf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_emb',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 6])]),
 'trf3': ColumnTransformer(transformers=[('scaling', MinMaxScaler(),
                                  slice(0, 10, None))]),
 'trf4': SelectKBest(k=8, score_func=<function chi2 at 0x7cab9b3ead40>),
 'trf5': DecisionTreeClassifier()}

In [17]:
prediction= pipe.predict(x_test)

In [34]:
prediction

array([0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1])

In [18]:
from sklearn.metrics import accuracy_score

In [19]:
pp=accuracy_score(y_test, prediction)

In [20]:
print(pp
  )

0.5952380952380952


In [21]:
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))

In [22]:
import os
print(os.listdir())

['.config', 'pipe.pkl', 'tested.csv', 'sample_data']
